# 00 · Entrenamiento del GPT decoder-only

Este notebook entrena un modelo GPT decoder-only en español con dos estilos:
- **Wiki**: textos enciclopédicos de Wikipedia en español
- **Poem**: poemas del dataset PoemasDelAlma

El modelo aprende a generar texto condicionado al estilo mediante un embedding de estilo.

## Estructura del notebook
1. Imports y configuración
2. Dataset de poemas
3. Tokenizer BPE
4. Datasets y DataLoaders
5. Definición del modelo (importado de `experiments/models/gpt_model.py`)
6. Bucle de entrenamiento
7. Generación de muestras


## 1. Imports y configuración

In [ ]:
import os, re, time, math, random, hashlib
from itertools import islice
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader
from torch.amp import autocast, GradScaler
from tqdm.auto import tqdm

from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.trainers import BpeTrainer
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

# Importamos el modelo desde experiments/
import sys
sys.path.insert(0, "..")
from models.gpt_model import GPT

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# Rutas
POEMS_CSV_PATH  = "../data/PoemasDelAlmaDataset.csv"
TOKENIZER_DIR   = "../tokenizer_mix_es"
TOKENIZER_JSON  = os.path.join(TOKENIZER_DIR, "tokenizer.json")
CHECKPOINT_DIR  = Path("../checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_TOKENIZER = True   # False para reusar tokenizer ya entrenado

# Semilla
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

### Hiperparámetros

In [ ]:
# Tokenizer
VOCAB_SIZE                  = 32_000
MIN_FREQ                    = 2
TOKENIZER_TRAIN_WIKI_ARTICLES = 60_000

# Modelo
block_size = 384
n_layers   = 8
d_model    = 512
n_heads    = 8
d_ff       = 2048
dropout    = 0.06

# Entrenamiento
batch_size       = 8
grad_accum_steps = 8
effective_batch  = batch_size * grad_accum_steps
base_lr          = 2e-4
betas            = (0.9, 0.95)
weight_decay     = 0.01
max_opt_steps    = 25_000
warmup_opt_steps = 800
log_every        = 50
eval_every       = 500

# Pesos por estilo
W_WIKI     = 1.00
W_POEM     = 0.70
LS_WIKI    = 0.03    # label smoothing wiki
LS_POEM    = 0.01    # label smoothing poem
STYLE_AUX_W = 0.12   # peso de la pérdida auxiliar de estilo

# Checkpoints
SAVE_EVERY_OPT_STEPS = 500
KEEP_LAST_N          = 5
SAVE_BEST_ON         = "wiki"   # métrica para guardar el mejor checkpoint

STYLE_WIKI = 0
STYLE_POEM = 1
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<wiki>", "<poem>", "<sep>", "<nl>"]

print(f"effective_batch: {effective_batch} secuencias")
print(f"tokens/opt_step: {effective_batch * block_size:,}")

## 2. Curriculum y utilidades de texto

In [ ]:
def p_poem_schedule(step, warm=3000, start=0.10, end=0.30):
    """Probabilidad de muestrear un poema en función del paso de entrenamiento."""
    if step <= 0: return start
    if step >= warm: return end
    return start + (end - start) * (step / warm)

CUT_HEADERS = [
    "Referencias", "Enlaces externos", "Véase también", "Bibliografía",
    "Notas", "Filmografía", "Discografía", "Premios", "Galería"
]

def clean_wiki_text(t):
    t = t.replace("\r", "\n")
    t = re.sub(r"\n{3,}", "\n\n", t)
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"[ \t]*\n[ \t]*", "\n", t).strip()
    for h in CUT_HEADERS:
        m = re.search(rf"\b{re.escape(h)}\b", t)
        if m and m.start() > 400:
            t = t[:m.start()].strip(); break
    t = re.sub(r"\bISBN\b.*?$", "", t, flags=re.IGNORECASE).strip()
    return t

def clean_poem_text(t):
    if t is None: return ""
    t = str(t).replace("\r", "\n")
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t).strip()
    return t

def text_to_tok_text(t):
    return t.replace("\n", " <nl> ")

def postprocess_decode(text):
    text = text.replace("<nl>", "\n")
    text = re.sub(r"[ \t]*\n[ \t]*", "\n", text)
    return text

def stable_hash(s):
    return int(hashlib.md5(s.encode("utf-8")).hexdigest(), 16)

def norm_for_hash(t):
    return re.sub(r"\s+", " ", t.lower().strip())

## 3. Dataset de poemas

In [ ]:
dfp = pd.read_csv(POEMS_CSV_PATH)
print("Columnas:", dfp.columns.tolist())
print("Total filas:", len(dfp))
dfp.head(3)

In [ ]:
def clean_poem_record(row):
    title  = str(row.get("Title",  "")).strip()
    poem   = str(row.get("Poem",   "")).strip()
    author = str(row.get("Author", "")).strip()
    if poem == "" or poem.lower() == "nan":
        return None
    poem = poem.replace("\r", "\n")
    poem = re.sub(r"[ \t]+", " ", poem)
    poem = re.sub(r"\n{3,}", "\n\n", poem).strip().strip('"').strip()
    parts = []
    if title  and title.lower()  != "nan": parts.append(f"Título: {title}")
    if author and author.lower() != "nan": parts.append(f"Autor: {author}")
    header = " | ".join(parts)
    return (header + "\n\n" + poem) if header else poem

raw_poems = [p for _, row in dfp.iterrows()
             if (p := clean_poem_record(row)) and len(p) >= 40]
print(f"Raw poems: {len(raw_poems)}")

# Deduplicación por hash
seen, poems_dedup = set(), []
for p in raw_poems:
    h = stable_hash(norm_for_hash(p))
    if h not in seen:
        seen.add(h); poems_dedup.append(p)
print(f"Dedup poems: {len(poems_dedup)}")

# Split 90/10 determinista por hash
poems_train, poems_val = [], []
for p in poems_dedup:
    r = (stable_hash(norm_for_hash(p)) % 10_000) / 10_000.0
    (poems_val if r < 0.10 else poems_train).append(p)
print(f"Train: {len(poems_train)} | Val: {len(poems_val)}")

## 4. Tokenizer BPE

In [ ]:
def train_tokenizer():
    os.makedirs(TOKENIZER_DIR, exist_ok=True)
    tmp_path = "mix_texts.txt"

    stream_tok = load_dataset("wikimedia/wikipedia", "20231101.es",
                              split="train", streaming=True)
    stream_tok = stream_tok.shuffle(buffer_size=10_000, seed=SEED)

    with open(tmp_path, "w", encoding="utf-8") as f:
        for ex in islice(stream_tok, TOKENIZER_TRAIN_WIKI_ARTICLES):
            t = ex.get("text", None)
            if not t: continue
            t = clean_wiki_text(t)
            if len(t) < 400: continue
            f.write(text_to_tok_text(t) + "\n")
        for p in poems_train:
            f.write(text_to_tok_text(clean_poem_text(p)) + "\n")

    tok = Tokenizer(BPE(unk_token=None))
    tok.pre_tokenizer = ByteLevel(add_prefix_space=True)
    tok.decoder = ByteLevelDecoder()
    trainer = BpeTrainer(vocab_size=VOCAB_SIZE, min_frequency=MIN_FREQ,
                         special_tokens=SPECIAL_TOKENS, show_progress=True)
    tok.train([tmp_path], trainer)
    tok.save(TOKENIZER_JSON)
    print("Tokenizer guardado:", TOKENIZER_JSON)
    return tok

if TRAIN_TOKENIZER or not os.path.exists(TOKENIZER_JSON):
    tokenizer = train_tokenizer()
else:
    tokenizer = Tokenizer.from_file(TOKENIZER_JSON)
    print("Tokenizer cargado:", TOKENIZER_JSON)

vocab = tokenizer.get_vocab()
pad_id = vocab["<pad>"]; bos_id = vocab["<bos>"]; eos_id = vocab["<eos>"]
wiki_tok_id = vocab["<wiki>"]; poem_tok_id = vocab["<poem>"]
sep_id = vocab["<sep>"]; nl_id = vocab["<nl>"]
vocab_size = tokenizer.get_vocab_size()
print(f"Vocab size: {vocab_size} | PAD:{pad_id} BOS:{bos_id} EOS:{eos_id}")

## 5. Datasets y DataLoaders

In [ ]:
def _make_prefix(style_idx):
    style_token = wiki_tok_id if style_idx == STYLE_WIKI else poem_tok_id
    return [bos_id, style_token, sep_id]

def _encode_text(text):
    return tokenizer.encode(text_to_tok_text(text)).ids

def _chunk_doc_ids(ids, block_size, stride=None, pad_to_full=True):
    if stride is None: stride = block_size
    if len(ids) < 2: return
    full_len = block_size + 1
    if len(ids) < full_len:
        if not pad_to_full: return
        padded = ids + [pad_id] * (full_len - len(ids))
        real_y = max(0, len(ids) - 1)
        mask = [1.0] * real_y + [0.0] * (block_size - real_y)
        yield (torch.tensor(padded[:-1], dtype=torch.long),
               torch.tensor(padded[1:],  dtype=torch.long),
               torch.tensor(mask, dtype=torch.float32))
        return
    last_start = len(ids) - full_len
    for start in range(0, last_start + 1, stride):
        chunk = ids[start:start + full_len]
        yield (torch.tensor(chunk[:-1], dtype=torch.long),
               torch.tensor(chunk[1:],  dtype=torch.long),
               torch.ones(block_size,   dtype=torch.float32))


def wiki_hash01(ex):
    h = hashlib.md5(str(ex.get("id", ex.get("title", ""))).encode()).hexdigest()
    return int(h, 16) % 10_000

class WikiDocsStream(IterableDataset):
    def __init__(self, hf_stream, block_size=384, max_articles=None,
                 min_chars=200, style_idx=STYLE_WIKI, split_fn=None,
                 stride=None, pad_to_full=True):
        self.hf_stream   = hf_stream
        self.block_size  = block_size
        self.max_articles = max_articles
        self.min_chars   = min_chars
        self.style_idx   = style_idx
        self.split_fn    = split_fn
        self.stride      = stride
        self.pad_to_full = pad_to_full

    def __iter__(self):
        n = 0
        for ex in self.hf_stream:
            if self.split_fn and not self.split_fn(ex): continue
            if self.max_articles and n >= self.max_articles: break
            t = ex.get("text", None)
            if not t: continue
            t = clean_wiki_text(t)
            if len(t) < self.min_chars: continue
            ids = _make_prefix(self.style_idx) + _encode_text(t) + [eos_id]
            n += 1
            for x, y, m in _chunk_doc_ids(ids, self.block_size,
                                           stride=self.stride,
                                           pad_to_full=self.pad_to_full):
                yield x, y, m, self.style_idx


class PoemDocsDataset(IterableDataset):
    def __init__(self, poems_list, block_size=384, style_idx=STYLE_POEM,
                 shuffle=True, seed=123, repeat=True, stride=None,
                 pad_to_full=True, min_chars=50):
        self.poems_list  = poems_list
        self.block_size  = block_size
        self.style_idx   = style_idx
        self.shuffle     = shuffle
        self.seed        = seed
        self.repeat      = repeat
        self.stride      = stride
        self.pad_to_full = pad_to_full
        self.min_chars   = min_chars

    def __iter__(self):
        worker = torch.utils.data.get_worker_info()
        wid = 0 if worker is None else worker.id
        rng = random.Random(self.seed + 9973 * wid)
        idxs = list(range(len(self.poems_list)))

        def one_pass():
            if self.shuffle: rng.shuffle(idxs)
            for i in idxs:
                t = self.poems_list[i]
                if not t or len(t) < self.min_chars: continue
                ids = _make_prefix(self.style_idx) + _encode_text(t) + [eos_id]
                for x, y, m in _chunk_doc_ids(ids, self.block_size,
                                               stride=self.stride,
                                               pad_to_full=self.pad_to_full):
                    yield x, y, m, self.style_idx

        if self.repeat:
            while True: yield from one_pass()
        else:
            yield from one_pass()


class MixedStreamCurriculum(IterableDataset):
    def __init__(self, wiki_iterable, poem_iterable, p_poem_schedule, seed=42):
        self.wiki_iterable  = wiki_iterable
        self.poem_iterable  = poem_iterable
        self.p_poem_schedule = p_poem_schedule
        self.seed           = seed

    def __iter__(self):
        worker = torch.utils.data.get_worker_info()
        wid = 0 if worker is None else worker.id
        rng = random.Random(self.seed + 10007 * wid)
        wiki_it = iter(self.wiki_iterable)
        poem_it = iter(self.poem_iterable)
        step = 0
        while True:
            p_poem = float(self.p_poem_schedule(step))
            x, y, m, sidx = next(poem_it) if rng.random() < p_poem else next(wiki_it)
            yield x, y, m, sidx
            step += 1


def collate_quad(batch):
    xs, ys, ms, ss = zip(*batch)
    return (torch.stack(xs), torch.stack(ys),
            torch.stack(ms), torch.tensor(ss, dtype=torch.long))

In [ ]:
wiki_stream_train = load_dataset("wikimedia/wikipedia", "20231101.es",
                                  split="train", streaming=True)
wiki_stream_train = wiki_stream_train.shuffle(buffer_size=5000, seed=123)
wiki_stream_val   = load_dataset("wikimedia/wikipedia", "20231101.es",
                                  split="train", streaming=True)
wiki_stream_val   = wiki_stream_val.shuffle(buffer_size=2000, seed=999)

stride = block_size // 2

wiki_train = WikiDocsStream(wiki_stream_train, block_size=block_size, min_chars=200,
    style_idx=STYLE_WIKI, split_fn=lambda ex: wiki_hash01(ex)/10000 >= 0.10,
    stride=stride, pad_to_full=True)

wiki_val   = WikiDocsStream(wiki_stream_val, block_size=block_size, max_articles=3000,
    min_chars=200, style_idx=STYLE_WIKI,
    split_fn=lambda ex: wiki_hash01(ex)/10000 < 0.10,
    stride=stride, pad_to_full=True)

poem_train = PoemDocsDataset(poems_train, block_size=block_size, style_idx=STYLE_POEM,
    shuffle=True, seed=11, repeat=True, stride=stride, pad_to_full=True, min_chars=50)

poem_val   = PoemDocsDataset(poems_val, block_size=block_size, style_idx=STYLE_POEM,
    shuffle=False, seed=22, repeat=False, stride=stride, pad_to_full=True, min_chars=1)

train_mix = MixedStreamCurriculum(wiki_train, poem_train,
    p_poem_schedule=lambda step: p_poem_schedule(step, warm=3000, start=0.10, end=0.30),
    seed=SEED)

train_loader     = DataLoader(train_mix, batch_size=batch_size, num_workers=0,
                              pin_memory=True, collate_fn=collate_quad)
wiki_val_loader  = DataLoader(wiki_val,  batch_size=batch_size, num_workers=0,
                              pin_memory=True, collate_fn=collate_quad)
poem_val_loader  = DataLoader(poem_val,  batch_size=batch_size, num_workers=0,
                              pin_memory=True, collate_fn=collate_quad)
print("DataLoaders OK")

## 6. Modelo y optimizador

In [ ]:
config_dict = dict(vocab_size=vocab_size, block_size=block_size,
                   n_layers=n_layers, d_model=d_model, n_heads=n_heads,
                   d_ff=d_ff, dropout=dropout, n_styles=2)
special_ids_dict = dict(pad_id=pad_id, bos_id=bos_id, eos_id=eos_id,
                        wiki_tok_id=wiki_tok_id, poem_tok_id=poem_tok_id,
                        sep_id=sep_id, nl_id=nl_id,
                        STYLE_WIKI=0, STYLE_POEM=1)

model = GPT(**config_dict).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parámetros: {n_params/1e6:.1f}M")

# Optimizador con weight decay selectivo
decay, no_decay = [], []
for name, p in model.named_parameters():
    if not p.requires_grad: continue
    (no_decay if p.dim() == 1 or name.endswith(".bias") else decay).append(p)

optimizer = torch.optim.AdamW(
    [{"params": decay,    "weight_decay": weight_decay},
     {"params": no_decay, "weight_decay": 0.0}],
    lr=base_lr, betas=betas, fused=(device.type == "cuda")
)

def lr_for_opt_step(s):
    if s < warmup_opt_steps:
        return base_lr * (s + 1) / warmup_opt_steps
    progress = (s - warmup_opt_steps) / max(1, max_opt_steps - warmup_opt_steps)
    cosine   = 0.5 * (1.0 + math.cos(math.pi * progress))
    return base_lr * (0.02 + 0.98 * cosine)

scaler = GradScaler(enabled=(device.type == "cuda"))

In [ ]:
def save_checkpoint(path, model, optimizer, scaler, opt_step, micro_step,
                    config_dict, tokenizer_json_path, special_ids_dict, extra_metrics=None):
    ckpt = dict(
        opt_step=opt_step, micro_step=micro_step,
        model_state=model.state_dict(),
        optimizer_state=optimizer.state_dict(),
        scaler_state=scaler.state_dict() if scaler else None,
        config=config_dict, tokenizer_json=str(tokenizer_json_path),
        special_ids=special_ids_dict, extra_metrics=extra_metrics or {},
    )
    torch.save(ckpt, path)

def rotate_checkpoints(prefix="step_", keep_last=5):
    files = sorted(CHECKPOINT_DIR.glob(f"{prefix}*.pt"))
    for f in files[:-keep_last]:
        try: f.unlink()
        except: pass

## 7. Bucle de entrenamiento

In [ ]:
def masked_ce_loss(logits, targets, mask, label_smoothing=0.0):
    B, T, V = logits.shape
    per_tok = F.cross_entropy(logits.view(-1, V), targets.view(-1),
                               reduction="none",
                               label_smoothing=label_smoothing).view(B, T)
    return (per_tok * mask).sum() / mask.sum().clamp_min(1.0)

@torch.no_grad()
def eval_loss(model, loader, n_batches=40, label_smoothing=0.0):
    model.eval()
    losses = []
    for x, y, m, style_idx in islice(loader, n_batches):
        x, y, m, style_idx = x.to(device), y.to(device), m.to(device), style_idx.to(device)
        with autocast(device_type="cuda", enabled=(device.type == "cuda")):
            logits, _ = model(x, style_idx=style_idx, capture=None,
                              pad_id=pad_id)
            losses.append(masked_ce_loss(logits, y, m, label_smoothing).item())
    model.train()
    return sum(losses) / len(losses)

In [ ]:
model.train()
optimizer.zero_grad(set_to_none=True)

micro_step = 0
opt_step   = 0
running    = 0.0
best_metric = float("inf")
t0 = time.time()

pbar = tqdm(total=max_opt_steps)

for batch in train_loader:
    x, y, m, style_idx = [t.to(device, non_blocking=True) for t in batch]

    is_poem = (style_idx == STYLE_POEM).float().view(-1, 1)
    w_style = W_POEM * is_poem + W_WIKI * (1.0 - is_poem)

    with autocast(device_type="cuda", enabled=(device.type == "cuda")):
        logits, caches, style_logits = model(
            x, style_idx=style_idx, capture=None,
            return_style_logits=True, pad_id=pad_id
        )
        B, T, V = logits.shape
        ce_w = F.cross_entropy(logits.view(-1,V), y.view(-1),
                                reduction="none", label_smoothing=LS_WIKI).view(B,T)
        ce_p = F.cross_entropy(logits.view(-1,V), y.view(-1),
                                reduction="none", label_smoothing=LS_POEM).view(B,T)
        loss_tok = ce_w * (1.0 - is_poem) + ce_p * is_poem
        lm_loss  = (loss_tok * m * w_style).sum() / (m * w_style).sum().clamp_min(1.0)
        style_loss = F.cross_entropy(style_logits, style_idx)
        loss = lm_loss + STYLE_AUX_W * style_loss

    scaler.scale(loss / grad_accum_steps).backward()
    micro_step += 1

    if micro_step % grad_accum_steps == 0:
        cur_lr = lr_for_opt_step(opt_step)
        for pg in optimizer.param_groups: pg["lr"] = cur_lr

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        optimizer.zero_grad(set_to_none=True)
        opt_step += 1; pbar.update(1)
        running  += loss.item()

        if opt_step % log_every == 0:
            pbar.set_description(f"step {opt_step} | loss {running/log_every:.3f} | lr {cur_lr:.2e}")
            running = 0.0

        if opt_step % eval_every == 0:
            v_wiki = eval_loss(model, wiki_val_loader, n_batches=16, label_smoothing=LS_WIKI)
            v_poem = eval_loss(model, poem_val_loader, n_batches=16, label_smoothing=LS_POEM)
            extra  = {"val_loss_wiki": v_wiki, "val_loss_poem": v_poem, "lr": cur_lr}

            save_checkpoint(CHECKPOINT_DIR/"latest.pt", model, optimizer, scaler,
                            opt_step, micro_step, config_dict, TOKENIZER_JSON,
                            special_ids_dict, extra)
            metric = v_wiki if SAVE_BEST_ON == "wiki" else v_poem
            if metric < best_metric:
                best_metric = metric
                save_checkpoint(CHECKPOINT_DIR/"best.pt", model, optimizer, scaler,
                                opt_step, micro_step, config_dict, TOKENIZER_JSON,
                                special_ids_dict, {**extra, "best_metric": best_metric})

            if opt_step % SAVE_EVERY_OPT_STEPS == 0:
                save_checkpoint(CHECKPOINT_DIR/f"step_{opt_step:06d}.pt", model, optimizer,
                                scaler, opt_step, micro_step, config_dict,
                                TOKENIZER_JSON, special_ids_dict, extra)
                rotate_checkpoints(keep_last=KEEP_LAST_N)

            ppl_w = math.exp(v_wiki) if v_wiki < 20 else float("inf")
            ppl_p = math.exp(v_poem) if v_poem < 20 else float("inf")
            print(f"\nstep {opt_step} | wiki loss {v_wiki:.3f} ppl {ppl_w:.1f}"
                  f" | poem loss {v_poem:.3f} ppl {ppl_p:.1f}")

        if opt_step >= max_opt_steps:
            break

pbar.close()
print(f"\nEntrenamiento completado en {(time.time()-t0)/60:.1f} min")

## 8. Generación de muestras

In [ ]:
def build_input_ids(prompt, style_idx):
    style_token = wiki_tok_id if style_idx == STYLE_WIKI else poem_tok_id
    prefix = [bos_id, style_token, sep_id]
    return prefix + tokenizer.encode(text_to_tok_text(prompt)).ids

@torch.no_grad()
def generate(model, prompt, style="wiki", max_new_tokens=160,
             temperature=0.9, top_p=0.95, top_k=0,
             presence_penalty=0.15, frequency_penalty=0.05,
             no_repeat_ngram_size=3):
    model.eval()
    style_idx = STYLE_WIKI if style == "wiki" else STYLE_POEM
    ids = build_input_ids(prompt, style_idx)
    x    = torch.tensor([ids], dtype=torch.long, device=device)
    sidx = torch.tensor([style_idx], dtype=torch.long, device=device)
    counts = torch.zeros(vocab_size, device=device)
    for tid in x[0]: counts[tid] += 1.0

    for _ in range(max_new_tokens):
        logits, _ = model(x, style_idx=sidx, capture=None,
                          pad_id=pad_id)
        nl = logits[0, -1].float() / temperature
        seen = (counts > 0).float()
        nl -= presence_penalty * seen + frequency_penalty * counts

        # top-p
        sl, si = torch.sort(nl, descending=True)
        cp = torch.cumsum(F.softmax(sl, dim=-1), dim=-1)
        cut = cp > top_p; cut[1:] = cut[:-1].clone(); cut[0] = False
        sl[cut] = float("-inf")
        nl2 = torch.empty_like(nl); nl2.scatter_(0, si, sl)

        probs = F.softmax(nl2, dim=-1)
        next_token = torch.multinomial(probs, 1)
        tid = next_token.item()
        x = torch.cat([x, next_token.view(1,1)], dim=1)
        counts[tid] += 1.0
        if tid == eos_id: break

    out = x[0].tolist()
    if len(out) >= 3 and out[0] == bos_id and out[2] == sep_id:
        out = out[3:]
    while out and out[-1] in (eos_id): out.pop()
    return postprocess_decode(tokenizer.decode(out))

print("=== WIKI ===")
print(generate(model, "La historia de España", style="wiki",
               temperature=0.75, top_p=0.92))
print("\n=== POEM ===")
print(generate(model, "En la noche silenciosa", style="poem",
               temperature=0.95, top_p=0.97))